## Data Preperation

In [56]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    fbeta_score,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold


In [57]:

PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv")

OUTPUT_DIR = PROJECT_ROOT / "Model_Results" / "prediction_model_optimisation_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(DATA_PATH)
print(df.shape)
df.head()

/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv
(69987, 56)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,diabetesMed,readmitted,readmitted_30,hba1c_group,primary_diagnosis,age_group,discharge_group,race_group,admission_source_group,medical_specialty_group
0,24437208,135,Caucasian,Female,[50-60),2,1,1,8,Cardiology,...,Yes,<30,1,No test was performed,Circulatory,30-60,Home,Caucasian,Physician/clinic referral,Cardiology
1,29758806,378,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,...,No,NO,0,No test was performed,Musculoskeletal,30-60,Home,Caucasian,Physician/clinic referral,Surgery
2,189899286,729,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,...,Yes,NO,0,Normal result of the test,Injury,>60,Other,Caucasian,Emergency room,Internal Medicine
3,64331490,774,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,...,Yes,NO,0,"High, medication changed",Other,>60,Home,Caucasian,Emergency room,Internal Medicine
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,...,Yes,NO,0,No test was performed,Genitourinary,30-60,Home,AfricanAmerican,Emergency room,Internal Medicine


In [58]:
# target_col = "readmitted_30"

# drop_cols = [
#     "readmitted",
#     "readmitted_30",
#     "encounter_id",
#     "patient_nbr"
# ]

# # Drop columns only if they actually exist
# drop_cols_existing = [col for col in drop_cols if col in df.columns]

# X = df.drop(columns=drop_cols_existing)
# y = df[target_col]

# print(X.shape)
# print(y.value_counts())
# print(y.value_counts(normalize=True))

target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

model_features = categorical_features + numeric_features

missing_features = [
    col for col in model_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"These modelling features are missing: {missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()


print("X shape:", X.shape)
print("Features used:", X.columns.tolist())
print("\nTarget distribution:")
print(y.value_counts())

forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = forbidden_features.intersection(X.columns)

assert not unexpected_features, (
    f"Unexpected features found in X: {unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names found in X."
)

print("Feature-selection checks passed.")

X shape: (69987, 18)
Features used: ['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target distribution:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64
Feature-selection checks passed.


In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, y_train.mean())
print("Test:", X_test.shape, y_test.mean())

# Split the original training set into a smaller training set and a validation set.
# The test set is not touched here.

X_model_train, X_val, y_model_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    stratify=y_train,
    random_state=42
)

print("Model training set:")
print(X_model_train.shape)
print(y_model_train.value_counts())
print(y_model_train.value_counts(normalize=True))

print("\nValidation set:")
print(X_val.shape)
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nFinal test set:")
print(X_test.shape)
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train: (55989, 18) 0.08980335423029524
Test: (13998, 18) 0.08979854264894985
Model training set:
(41991, 18)
readmitted_30
0    38220
1     3771
Name: count, dtype: int64
readmitted_30
0    0.910195
1    0.089805
Name: proportion, dtype: float64

Validation set:
(13998, 18)
readmitted_30
0    12741
1     1257
Name: count, dtype: int64
readmitted_30
0    0.910201
1    0.089799
Name: proportion, dtype: float64

Final test set:
(13998, 18)
readmitted_30
0    12741
1     1257
Name: count, dtype: int64
readmitted_30
0    0.910201
1    0.089799
Name: proportion, dtype: float64


In [60]:
# categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
# numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()

# print("Categorical features:", len(categorical_features))
# print(categorical_features)

# print("Numeric features:", len(numeric_features))
# print(numeric_features)

categorical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

numeric_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "num",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

In [61]:
def save_confusion_matrix(model, X_test, y_test, model_name, output_dir):
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual not readmitted", "Actual readmitted"],
        columns=["Predicted not readmitted", "Predicted readmitted"]
    )

    file_name = (
        model_name
        .lower()
        .replace(":", "")
        .replace(" ", "_")
        .replace("/", "_")
    )

    cm_df.to_csv(output_dir / f"confusion_matrix_{file_name}.csv")

    tn, fp, fn, tp = cm.ravel()

    total = tn + fp + fn + tp

    cm_long = pd.DataFrame([{
        "model": model_name,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "total": total,
        "true_negative_rate": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "false_positive_rate": fp / (tn + fp) if (tn + fp) > 0 else 0,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0,
        "true_positive_rate_recall": tp / (fn + tp) if (fn + tp) > 0 else 0
    }])

    return cm_df, cm_long

def evaluate_predictions_from_proba(
    y_true,
    y_proba,
    threshold=0.5,
    model_name="Model"
):
    """
    Evaluate binary predictions created from predicted probabilities.

    Parameters
    ----------
    y_true:
        True labels, where 0 means not readmitted and
        1 means readmitted within 30 days.

    y_proba:
        Predicted probability of class 1.

    threshold:
        Probability threshold used to create class predictions.

    model_name:
        Name stored in the results table.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    specificity = (
        tn / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_positive_rate = (
        fp / actual_negative
        if actual_negative > 0
        else np.nan
    )

    false_negative_rate = (
        fn / actual_positive
        if actual_positive > 0
        else np.nan
    )

    predicted_positive_rate = (
        predicted_positive / total
        if total > 0
        else np.nan
    )

    patients_flagged_per_true_readmission = (
        predicted_positive / tp
        if tp > 0
        else np.nan
    )

    results = {
        "model": model_name,
        "threshold": float(threshold),

        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f2": fbeta_score(
            y_true,
            y_pred,
            beta=2,
            zero_division=0
        ),

        # These probability-based metrics do not depend
        # on the classification threshold.
        "auroc": roc_auc_score(y_true, y_proba),
        "auprc": average_precision_score(
            y_true,
            y_proba
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_proba
        ),

        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),

        "predicted_positive": int(predicted_positive),
        "predicted_negative": int(predicted_negative),
        "predicted_positive_rate": predicted_positive_rate,

        "patients_flagged_per_true_readmission_found": (
            patients_flagged_per_true_readmission
        )
    }

    return results


def confusion_matrix_from_proba(
    y_true,
    y_proba,
    threshold=0.5
):
    """
    Create a labelled confusion matrix from probabilities.
    """

    y_pred = (np.asarray(y_proba) >= threshold).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    return pd.DataFrame(
        cm,
        index=[
            "Actual not readmitted",
            "Actual readmitted"
        ],
        columns=[
            "Predicted not readmitted",
            "Predicted readmitted"
        ]
    )


def threshold_sweep(
    y_true,
    y_proba,
    model_name="Model",
    thresholds=None
):
    """
    Create a descriptive table showing performance
    across a range of probability thresholds.

    This table is useful for inspection and plotting.
    It is not used for final threshold selection.
    """

    if thresholds is None:
        thresholds = np.round(
            np.arange(0.05, 0.951, 0.01),
            2
        )

    results = [
        evaluate_predictions_from_proba(
            y_true=y_true,
            y_proba=y_proba,
            threshold=threshold,
            model_name=model_name
        )
        for threshold in thresholds
    ]

    return pd.DataFrame(results)


def choose_threshold_for_minimum_recall(
    y_true,
    y_proba,
    min_recall=0.80,
    model_name="Model"
):
    """
    Select the threshold with the lowest false-positive rate
    among thresholds achieving the required recall.

    Selection rule:
        1. Recall must be at least min_recall.
        2. Minimise the false-positive rate.
        3. If tied, use the highest threshold.
    """

    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    false_positive_rates, recalls, thresholds = roc_curve(
        y_true,
        y_proba,
        drop_intermediate=False
    )

    candidate_table = pd.DataFrame({
        "threshold": thresholds,
        "recall": recalls,
        "false_positive_rate": false_positive_rates,
        "specificity": 1 - false_positive_rates
    })

    # The first ROC threshold can be infinity.
    candidate_table = candidate_table[
        np.isfinite(candidate_table["threshold"])
    ].copy()

    eligible_candidates = candidate_table[
        candidate_table["recall"] >= min_recall
    ].copy()

    if eligible_candidates.empty:
        raise ValueError(
            f"No threshold achieved recall >= {min_recall:.2f}."
        )

    eligible_candidates = eligible_candidates.sort_values(
        by=[
            "false_positive_rate",
            "threshold"
        ],
        ascending=[
            True,
            False
        ]
    ).reset_index(drop=True)

    selected_threshold = float(
        eligible_candidates.iloc[0]["threshold"]
    )

    selected_metrics = evaluate_predictions_from_proba(
        y_true=y_true,
        y_proba=y_proba,
        threshold=selected_threshold,
        model_name=model_name
    )

    return (
        selected_threshold,
        selected_metrics,
        eligible_candidates
    )

## Regularised Logistic Regression optimisation

The model-training subset is used for hyperparameter tuning with
stratified cross-validation. Average precision, equivalent to AUPRC,
is used as the main cross-validation score because the outcome is
strongly imbalanced.

The validation set is then used to choose a classification threshold.
The threshold must achieve at least 80% recall for readmitted patients.
Among eligible thresholds, the threshold with the lowest false-positive
rate is selected.

The final test set is used once, after the model specification and
threshold have both been fixed.

In [62]:
RECALL_TARGET = 0.80

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

regularised_log_reg_param_grid = {
    "model__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0
    ],

    # scikit-learn 1.8 uses l1_ratio instead of
    # the deprecated penalty parameter.
    #
    # 0.0 = pure L2
    # 1.0 = pure L1
    # values between them = Elastic Net
    "model__l1_ratio": [
        0.0,
        0.25,
        0.5,
        0.75,
        1.0
    ],

    # Test whether class weighting improves ranking.
    "model__class_weight": [
        None,
        "balanced"
    ]
}

In [63]:
additive_log_reg_pipeline = Pipeline(steps=[
    (
        "preprocess",
        preprocess
    ),
    (
        "model",
        LogisticRegression(
            solver="saga",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        )
    )
])

additive_log_reg_search = GridSearchCV(
    estimator=additive_log_reg_pipeline,
    param_grid=regularised_log_reg_param_grid,
    scoring="average_precision",
    cv=cross_validation,
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

additive_log_reg_search.fit(
    X_model_train,
    y_model_train
)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=0.001))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None, 'balanced'], 'model__l1_ratio': [0.0, 0.25, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting the

In [64]:
print(
    "Best additive Logistic Regression parameters:"
)
print(additive_log_reg_search.best_params_)

print(
    "\nBest cross-validation AUPRC:"
)
print(additive_log_reg_search.best_score_)

additive_cv_results = pd.DataFrame(
    additive_log_reg_search.cv_results_
)

additive_cv_results_selected = (
    additive_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "param_model__C",
            "param_model__l1_ratio",
            "param_model__class_weight"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

additive_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_additive_cv_results.csv",
    index=False
)

additive_cv_results_selected.head(10)

Best additive Logistic Regression parameters:
{'model__C': 0.01, 'model__class_weight': None, 'model__l1_ratio': 0.0}

Best cross-validation AUPRC:
0.1514750550581429


,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_model__C,param_model__l1_ratio,param_model__class_weight
0,1,0.151475,0.006474,0.153244,0.01,0.00,None
1,2,0.151449,0.006259,0.153360,0.10,0.75,None
2,3,0.151442,0.006208,0.153563,0.10,0.50,None
3,4,0.151339,0.006082,0.153735,0.10,0.00,None
4,5,0.151319,0.006287,0.153106,0.10,1.00,None
5,6,0.151319,0.006223,0.153719,0.10,0.25,None
6,7,0.151284,0.005980,0.153760,1.00,0.75,None
7,8,0.151273,0.005961,0.153756,1.00,0.50,None
8,9,0.151262,0.005942,0.153768,1.00,0.00,None
9,10,0.151260,0.005999,0.153759,1.00,1.00,None


In [65]:
additive_log_reg_model = (
    additive_log_reg_search.best_estimator_
)

y_val_proba_additive = (
    additive_log_reg_model
    .predict_proba(X_val)[:, 1]
)

In [66]:
additive_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_additive,
        threshold=0.5,
        model_name=(
            "Regularised Logistic Regression "
            "additive validation default"
        )
    )
)

(
    additive_selected_threshold,
    additive_val_selected_results,
    additive_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_additive,
    min_recall=RECALL_TARGET,
    model_name=(
        "Regularised Logistic Regression "
        "additive validation selected"
    )
)

print(
    "Selected additive-model threshold:",
    additive_selected_threshold
)

pd.DataFrame([
    additive_val_default_results,
    additive_val_selected_results
])

Selected additive-model threshold: 0.06740764600230212


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression additive valid...,0.500000,0.909844,0.142857,0.000796,0.999529,0.000471,0.999204,0.001582,0.000993,...,0.13586,0.080635,12735,6,1256,1,7,13991,0.000500,7.00000
1,Regularised Logistic Regression additive valid...,0.067408,0.400057,0.109909,0.800318,0.360568,0.639432,0.199682,0.193276,0.354700,...,0.13586,0.080635,4594,8147,251,1006,9153,4845,0.653879,9.09841


In [67]:
additive_eligible_thresholds.head(10)

,threshold,recall,false_positive_rate,specificity
0,0.067408,0.800318,0.639432,0.360568
1,0.067404,0.801114,0.639432,0.360568
2,0.067401,0.801114,0.639510,0.360490
3,0.067398,0.801114,0.639589,0.360411
4,0.067397,0.801114,0.639667,0.360333
5,0.067393,0.801114,0.639746,0.360254
6,0.067390,0.801114,0.639824,0.360176
7,0.067390,0.801114,0.639903,0.360097
8,0.067380,0.801909,0.639903,0.360097
9,0.067372,0.801909,0.639981,0.360019


In [68]:
additive_threshold_sweep = threshold_sweep(
    y_true=y_val,
    y_proba=y_val_proba_additive,
    model_name=(
        "Regularised Logistic Regression additive validation"
    )
)

additive_threshold_sweep.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_additive_validation_threshold_sweep.csv",
    index=False
)

additive_threshold_sweep[
    [
        "threshold",
        "recall",
        "precision",
        "specificity",
        "false_positive_rate",
        "f2",
        "false_positive",
        "false_negative"
    ]
].sort_values(
    "threshold",
    ascending=False
)

,threshold,recall,precision,specificity,false_positive_rate,f2,false_positive,false_negative
90,0.95,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
89,0.94,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
88,0.93,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
87,0.92,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
86,0.91,0.000000,0.000000,1.000000,0.000000,0.000000,0,1257
...,...,...,...,...,...,...,...,...
4,0.09,0.556881,0.125313,0.616514,0.383486,0.329753,4886,557
3,0.08,0.642005,0.119220,0.532062,0.467938,0.342036,5962,450
2,0.07,0.765314,0.112004,0.401381,0.598619,0.353235,7627,295
1,0.06,0.894193,0.102396,0.226670,0.773330,0.351140,9853,133


In [69]:
additive_val_selected_cm = (
    confusion_matrix_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_additive,
        threshold=additive_selected_threshold
    )
)

additive_val_selected_cm.to_csv(
    OUTPUT_DIR
    / "confusion_matrix_regularised_logistic_regression_additive_validation.csv"
)

additive_val_selected_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4594,8147
Actual readmitted,251,1006


In [ ]:
# create classifiction report for the additive model on the test set using the selected threshold
y_additive_test_proba = (
    additive_log_reg_model
    .predict_proba(X_test)[:, 1]
)

print(classification_report(
    y_test,
    (y_additive_test_proba >= additive_selected_threshold).astype(int),
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.95      0.36      0.52     12741
    Readmitted       0.11      0.81      0.19      1257

      accuracy                           0.40     13998
     macro avg       0.53      0.58      0.36     13998
  weighted avg       0.88      0.40      0.49     13998



In [71]:
additive_validation_comparison = pd.DataFrame([
    additive_val_default_results,
    additive_val_selected_results
])

additive_validation_comparison.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_additive_validation_results.csv",
    index=False
)

additive_validation_comparison

,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression additive valid...,0.500000,0.909844,0.142857,0.000796,0.999529,0.000471,0.999204,0.001582,0.000993,...,0.13586,0.080635,12735,6,1256,1,7,13991,0.000500,7.00000
1,Regularised Logistic Regression additive valid...,0.067408,0.400057,0.109909,0.800318,0.360568,0.639432,0.199682,0.193276,0.354700,...,0.13586,0.080635,4594,8147,251,1006,9153,4845,0.653879,9.09841


### Interaction experiment
HbA1c group × primary diagnosis interaction is tested below

In [72]:
INTERACTION_FEATURE = (
    "hba1c_x_primary_diagnosis"
)


def add_hba1c_diagnosis_interaction(X_data):
    """
    Add a categorical feature representing every combination
    of HbA1c group and primary diagnosis.
    """

    X_with_interaction = X_data.copy()

    hba1c_values = (
        X_with_interaction["hba1c_group"]
        .astype("string")
        .fillna("Missing")
    )

    diagnosis_values = (
        X_with_interaction["primary_diagnosis"]
        .astype("string")
        .fillna("Missing")
    )

    X_with_interaction[INTERACTION_FEATURE] = (
        hba1c_values
        + " | "
        + diagnosis_values
    )

    return X_with_interaction


X_model_train_interaction = (
    add_hba1c_diagnosis_interaction(
        X_model_train
    )
)

X_val_interaction = (
    add_hba1c_diagnosis_interaction(
        X_val
    )
)

X_test_interaction = (
    add_hba1c_diagnosis_interaction(
        X_test
    )
)

interaction_categorical_features = (
    categorical_features
    + [INTERACTION_FEATURE]
)

print(
    X_model_train_interaction[
        [
            "hba1c_group",
            "primary_diagnosis",
            INTERACTION_FEATURE
        ]
    ].head()
)

                    hba1c_group primary_diagnosis  \
38996     No test was performed   Musculoskeletal   
40867  High, medication changed             Other   
25126     No test was performed         Digestive   
2049      No test was performed         Digestive   
3304      No test was performed   Musculoskeletal   

                     hba1c_x_primary_diagnosis  
38996  No test was performed | Musculoskeletal  
40867         High, medication changed | Other  
25126        No test was performed | Digestive  
2049         No test was performed | Digestive  
3304   No test was performed | Musculoskeletal  


In [73]:
interaction_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

interaction_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

interaction_preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            interaction_categorical_transformer,
            interaction_categorical_features
        ),
        (
            "num",
            interaction_numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

In [ ]:
interaction_log_reg_pipeline = Pipeline(steps=[
    (
        "preprocess",
        interaction_preprocess
    ),
    (
        "model",
        LogisticRegression(
            solver="saga",
            max_iter=5000,
            tol=1e-3,
            random_state=42
        )
    )
])

interaction_log_reg_search = GridSearchCV(
    estimator=interaction_log_reg_pipeline,
    param_grid=regularised_log_reg_param_grid,
    scoring="average_precision",
    cv=cross_validation,
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

interaction_log_reg_search.fit(
    X_model_train_interaction,
    y_model_train
)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=0.001))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None, 'balanced'], 'model__l1_ratio': [0.0, 0.25, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting the

In [75]:
print(
    "Best interaction Logistic Regression parameters:"
)
print(interaction_log_reg_search.best_params_)

print(
    "\nBest interaction cross-validation AUPRC:"
)
print(interaction_log_reg_search.best_score_)

interaction_cv_results = pd.DataFrame(
    interaction_log_reg_search.cv_results_
)

interaction_cv_results_selected = (
    interaction_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "param_model__C",
            "param_model__l1_ratio",
            "param_model__class_weight"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

interaction_cv_results_selected.to_csv(
    OUTPUT_DIR
    / "regularised_logistic_regression_interaction_cv_results.csv",
    index=False
)

interaction_cv_results_selected.head(10)

Best interaction Logistic Regression parameters:
{'model__C': 1.0, 'model__class_weight': None, 'model__l1_ratio': 0.0}

Best interaction cross-validation AUPRC:
0.15219259635008467


,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_model__C,param_model__l1_ratio,param_model__class_weight
0,1,0.152193,0.005554,0.155623,1.0,0.00,None
1,2,0.152161,0.005592,0.155590,1.0,0.25,None
2,3,0.152160,0.005651,0.155538,1.0,0.50,None
3,4,0.152143,0.005488,0.155697,10.0,0.00,None
4,5,0.152137,0.005495,0.155690,10.0,0.50,None
5,6,0.152136,0.005488,0.155690,10.0,0.25,None
6,7,0.152133,0.005507,0.155676,10.0,1.00,None
7,8,0.152131,0.005500,0.155683,10.0,0.75,None
8,9,0.152102,0.005935,0.155223,0.1,0.00,None
9,10,0.152096,0.005688,0.155481,1.0,0.75,None


In [76]:
interaction_log_reg_model = (
    interaction_log_reg_search.best_estimator_
)

y_val_proba_interaction = (
    interaction_log_reg_model
    .predict_proba(X_val_interaction)[:, 1]
)

interaction_val_default_results = (
    evaluate_predictions_from_proba(
        y_true=y_val,
        y_proba=y_val_proba_interaction,
        threshold=0.5,
        model_name=(
            "Regularised Logistic Regression "
            "interaction validation default"
        )
    )
)

(
    interaction_selected_threshold,
    interaction_val_selected_results,
    interaction_eligible_thresholds
) = choose_threshold_for_minimum_recall(
    y_true=y_val,
    y_proba=y_val_proba_interaction,
    min_recall=RECALL_TARGET,
    model_name=(
        "Regularised Logistic Regression "
        "interaction validation selected"
    )
)

print(
    "Selected interaction-model threshold:",
    interaction_selected_threshold
)

pd.DataFrame([
    interaction_val_default_results,
    interaction_val_selected_results
])

Selected interaction-model threshold: 0.06580223068940529


,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression interaction va...,0.500000,0.909773,0.12500,0.000796,0.999451,0.000549,0.999204,0.001581,0.000993,...,0.134439,0.080724,12734,7,1256,1,8,13990,0.000572,8.000000
1,Regularised Logistic Regression interaction va...,0.065802,0.389484,0.10816,0.800318,0.348952,0.651048,0.199682,0.190566,0.351036,...,0.134439,0.080724,4446,8295,251,1006,9301,4697,0.664452,9.245527


In [77]:
logistic_regression_validation_comparison = pd.DataFrame([
    additive_val_selected_results,
    interaction_val_selected_results
])

comparison_columns = [
    "model",
    "threshold",
    "auprc",
    "auroc",
    "recall",
    "precision",
    "specificity",
    "false_positive_rate",
    "f2",
    "true_positive",
    "false_positive",
    "false_negative",
    "predicted_positive_rate",
    "patients_flagged_per_true_readmission_found"
]

logistic_regression_validation_comparison = (
    logistic_regression_validation_comparison[
        comparison_columns
    ]
)

logistic_regression_validation_comparison.to_csv(
    OUTPUT_DIR
    / "logistic_regression_candidate_validation_comparison.csv",
    index=False
)

logistic_regression_validation_comparison

,model,threshold,auprc,auroc,recall,precision,specificity,false_positive_rate,f2,true_positive,false_positive,false_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Regularised Logistic Regression additive valid...,0.067408,0.135860,0.621407,0.800318,0.109909,0.360568,0.639432,0.354700,1006,8147,251,0.653879,9.098410
1,Regularised Logistic Regression interaction va...,0.065802,0.134439,0.617725,0.800318,0.108160,0.348952,0.651048,0.351036,1006,8295,251,0.664452,9.245527


In [78]:
eligible_logistic_models = (
    logistic_regression_validation_comparison[
        logistic_regression_validation_comparison[
            "recall"
        ] >= RECALL_TARGET
    ]
    .copy()
)

if eligible_logistic_models.empty:
    raise ValueError(
        "No Logistic Regression candidate achieved "
        f"validation recall >= {RECALL_TARGET:.2f}."
    )

selected_logistic_row = (
    eligible_logistic_models
    .sort_values(
        by=[
            "false_positive_rate",
            "precision",
            "auprc"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .iloc[0]
)

selected_logistic_name = (
    selected_logistic_row["model"]
)

print(
    "Selected Logistic Regression candidate:"
)
print(selected_logistic_name)

selected_logistic_row

Selected Logistic Regression candidate:
Regularised Logistic Regression additive validation selected


model                                          Regularised Logistic Regression additive valid...
threshold                                                                               0.067408
auprc                                                                                    0.13586
auroc                                                                                   0.621407
recall                                                                                  0.800318
precision                                                                               0.109909
specificity                                                                             0.360568
false_positive_rate                                                                     0.639432
f2                                                                                        0.3547
true_positive                                                                               1006
false_positive                

In [79]:
logistic_model_registry = {
    (
        "Regularised Logistic Regression "
        "additive validation selected"
    ): {
        "model": additive_log_reg_model,
        "threshold": additive_selected_threshold,
        "X_test": X_test,
        "short_name": "additive"
    },

    (
        "Regularised Logistic Regression "
        "interaction validation selected"
    ): {
        "model": interaction_log_reg_model,
        "threshold": interaction_selected_threshold,
        "X_test": X_test_interaction,
        "short_name": "interaction"
    }
}

selected_logistic_configuration = (
    logistic_model_registry[
        selected_logistic_name
    ]
)

final_logistic_model = (
    selected_logistic_configuration["model"]
)

final_logistic_threshold = (
    selected_logistic_configuration["threshold"]
)

final_logistic_X_test = (
    selected_logistic_configuration["X_test"]
)

final_logistic_short_name = (
    selected_logistic_configuration["short_name"]
)

print(
    "Final threshold:",
    final_logistic_threshold
)

Final threshold: 0.06740764600230212


In [80]:
y_test_proba_final_logistic = (
    final_logistic_model
    .predict_proba(final_logistic_X_test)[:, 1]
)

final_logistic_test_results = (
    evaluate_predictions_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_final_logistic,
        threshold=final_logistic_threshold,
        model_name=(
            "Final Regularised Logistic Regression test"
        )
    )
)

pd.Series(final_logistic_test_results)

model                                          Final Regularised Logistic Regression test
threshold                                                                        0.067408
accuracy                                                                         0.398771
precision                                                                        0.110797
recall                                                                            0.81066
specificity                                                                      0.358135
false_positive_rate                                                              0.641865
false_negative_rate                                                               0.18934
f1                                                                               0.194949
f2                                                                               0.358172
auroc                                                                            0.637366
auprc     

In [81]:
final_logistic_test_cm = (
    confusion_matrix_from_proba(
        y_true=y_test,
        y_proba=y_test_proba_final_logistic,
        threshold=final_logistic_threshold
    )
)

final_logistic_test_cm

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,4563,8178
Actual readmitted,238,1019


In [82]:
final_logistic_test_results_df = pd.DataFrame([
    final_logistic_test_results
])

final_logistic_test_results_df.to_csv(
    OUTPUT_DIR
    / "final_regularised_logistic_regression_test_metrics.csv",
    index=False
)

final_logistic_test_cm.to_csv(
    OUTPUT_DIR
    / "final_regularised_logistic_regression_test_confusion_matrix.csv"
)

final_logistic_test_results_df

,model,threshold,accuracy,precision,recall,specificity,false_positive_rate,false_negative_rate,f1,f2,...,auprc,brier_score,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_negative,predicted_positive_rate,patients_flagged_per_true_readmission_found
0,Final Regularised Logistic Regression test,0.067408,0.398771,0.110797,0.81066,0.358135,0.641865,0.18934,0.194949,0.358172,...,0.149858,0.079984,4563,8178,238,1019,9197,4801,0.657022,9.025515


In [83]:
y_test_pred_final_logistic = (
    y_test_proba_final_logistic
    >= final_logistic_threshold
).astype(int)

print(
    classification_report(
        y_test,
        y_test_pred_final_logistic,
        target_names=[
            "Not readmitted",
            "Readmitted"
        ],
        zero_division=0
    )
)

                precision    recall  f1-score   support

Not readmitted       0.95      0.36      0.52     12741
    Readmitted       0.11      0.81      0.19      1257

      accuracy                           0.40     13998
     macro avg       0.53      0.58      0.36     13998
  weighted avg       0.88      0.40      0.49     13998



Adding the specific HbA1c × primary-diagnosis interaction did not improve this model's ability to predict individual 30-day readmissions on the validation data.

Hyperparameter tuning selected an L2-regularised additive Logistic Regression model without class weighting. A paper-informed interaction between HbA1c group and primary diagnosis was also evaluated but did not improve validation performance. At a validation-selected threshold of 0.0674, the additive model achieved 81.1% recall on the final test set, satisfying the required 80% target. However, this required flagging 65.7% of patients and produced a false-positive rate of 64.2%, indicating limited practical discrimination.